In [27]:
import torch.nn as nn
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [28]:
VOCAB_SIZE  = 30000
SEQ_LEN     = 8      # T — context window; input sequence is (T, D_MODEL) = (8, 8)
D_MODEL     = 8      # input hidden dim

# ── MLA-specific dims ─────────────────────────────────────────────────────────
N_HEADS      = 2     # number of Q/K heads
D_HEAD_NOPE  = 4     # NoPE (content) dim per head
D_HEAD_ROPE  = 2     # RoPE (positional) dim per head — small, stays uncompressed
D_HEAD       = D_HEAD_NOPE + D_HEAD_ROPE   # 6: effective full head dim for attention
D_C_KV       = 4    # KV latent compression dim  →  W_DKV: (D_MODEL × D_C_KV) = (4×4)
D_C_Q        = 4    # Q  latent compression dim  →  W_DQ:  (D_MODEL × D_C_Q)  = (4×4)

D_FF        = 16
DROPOUT     = 0.1
BATCH_SIZE  = 64
LR          = 3e-4
EPOCHS      = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
print(f"input shape per sequence: ({SEQ_LEN}, {D_MODEL})")
print(f"W_DKV shape: ({D_MODEL}, {D_C_KV})")

device: cuda
input shape per sequence: (8, 8)
W_DKV shape: (8, 4)


In [29]:
TRAIN_BIN = Path.cwd() / "data" / "train.bin"

token_ids = np.fromfile(TRAIN_BIN, dtype=np.uint16).astype(np.int64)
print(f"total tokens: {len(token_ids):,}")

class TokenDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.ids) - self.seq_len

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.seq_len]
        y = self.ids[idx + 1 : idx + self.seq_len + 1]
        return x, y

dataset    = TokenDataset(token_ids, SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"batches per epoch: {len(dataloader)}")

total tokens: 136,633
batches per epoch: 2134


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Decoupled RoPE utilities
# Applied ONLY to the small Q_rope / K_rope pathways, never to the NoPE content.
# ══════════════════════════════════════════════════════════════════════════════

def precompute_freqs_cis(d_rope: int, max_seq_len: int, theta: float = 10000.0):
    """
    Returns complex unit vectors for RoPE rotation.
    d_rope must be even.
    Output shape: (max_seq_len, d_rope // 2)  — complex64
    """
    assert d_rope % 2 == 0
    # inverse frequencies: θ_i = 1 / 10000^(2i / d_rope)
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_rope, 2).float() / d_rope))
    t        = torch.arange(max_seq_len, dtype=torch.float32)
    # outer product → angles matrix  (max_seq_len, d_rope//2)
    angles   = torch.outer(t, inv_freq)
    # e^{i·angle} = cos + i·sin  — the rotation in complex form
    freqs_cis = torch.polar(torch.ones_like(angles), angles)
    return freqs_cis   # (T, d_rope//2)  complex64


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    Rotate x using pre-computed complex frequencies.
    x         : (B, n_heads, T, d_rope)   — real
    freqs_cis : (T, d_rope // 2)          — complex
    Returns   : (B, n_heads, T, d_rope)   — real, rotated
    """
    B, H, T, d = x.shape
    # pair up consecutive dims as Re/Im of a complex number
    x_c = torch.view_as_complex(x.float().reshape(B, H, T, d // 2, 2))  # (B,H,T,d/2) complex
    # broadcast freqs over batch and heads
    f   = freqs_cis[:T].unsqueeze(0).unsqueeze(0)    # (1, 1, T, d/2) complex
    # multiply = rotate in 2D plane
    out = torch.view_as_real(x_c * f).reshape(B, H, T, d)
    return out.to(x.dtype)

# Pre-compute once for the full context length
freqs_cis = precompute_freqs_cis(D_HEAD_ROPE, SEQ_LEN).to(DEVICE)
print(f"freqs_cis shape: {freqs_cis.shape}  (T={SEQ_LEN}, d_rope/2={D_HEAD_ROPE//2})  dtype={freqs_cis.dtype}")

freqs_cis shape: torch.Size([8, 1])  (T=8, d_rope/2=1)  dtype=torch.complex64


# RoPE Demo

In [36]:
cis = precompute_freqs_cis(4, SEQ_LEN)
x = torch.ones(1, 1, SEQ_LEN, 4)  # (B,H,T,d_rope)
out = apply_rope(x, cis)
print("Input tensor:")
print(x)
print("\nPre-computed freqs_cis:")
print(cis)
print("\nRotated tensor:")
print(out)

Input tensor:
tensor([[[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]]])

Pre-computed freqs_cis:
tensor([[ 1.0000+0.0000j,  1.0000+0.0000j],
        [ 0.5403+0.8415j,  0.9999+0.0100j],
        [-0.4161+0.9093j,  0.9998+0.0200j],
        [-0.9900+0.1411j,  0.9996+0.0300j],
        [-0.6536-0.7568j,  0.9992+0.0400j],
        [ 0.2837-0.9589j,  0.9988+0.0500j],
        [ 0.9602-0.2794j,  0.9982+0.0600j],
        [ 0.7539+0.6570j,  0.9976+0.0699j]])

Rotated tensor:
tensor([[[[ 1.0000,  1.0000,  1.0000,  1.0000],
          [-0.3012,  1.3818,  0.9900,  1.0099],
          [-1.3254,  0.4932,  0.9798,  1.0198],
          [-1.1311, -0.8489,  0.9696,  1.0295],
          [ 0.1032, -1.4104,  0.9592,  1.0392],
          [ 1.2426, -0.6753,  0.9488,  1.0487],
          [ 1.2396,  0.6808,  0.9382,  1.0582],
          [ 0.0969,  1.4109,